In [1]:
import pandas as pd
import numpy as np
import os
import gc
import random
import warnings
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import KFold
import math

warnings.filterwarnings('ignore')

# ==============================================================================
# 1. 設定 (Configuration)
# ==============================================================================
# データセットのパス（Kaggle環境を想定）
if os.path.exists('/kaggle/input/joai-competition-2026/train.csv'):
    INPUT_DIR = '/kaggle/input/joai-competition-2026'
elif os.path.exists('/kaggle/input/ai2026/train.csv'):
    INPUT_DIR = '/kaggle/input/ai2026'
else:
    INPUT_DIR = '.' # ローカルまたはアップロード済みの場合

TRAIN_PATH = os.path.join(INPUT_DIR, 'train.csv')
TEST_PATH = os.path.join(INPUT_DIR, 'test.csv')
SUBMISSION_PATH = 'submission_pid_dl.csv'

# ハイパーパラメータ
BATCH_SIZE = 64
N_FOLDS = 5
EPOCHS = 20
LEARNING_RATE = 1e-3
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using Device: {DEVICE}")

def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True

seed_everything(42)

# ==============================================================================
# 2. 特徴量エンジニアリング (PID Features)
# ==============================================================================
def add_pid_features(df):
    """
    PID制御の概念（比例・積分・微分）に基づいた特徴量を追加
    """
    df_eng = df.copy()
    
    # IDやターゲット以外の「脳活動データ列」を特定
    ignore_cols = ['id', 'sample_id', 'day_n', 'time', 'lever', 'mouse_id']
    base_cols = [c for c in df_eng.columns if c not in ignore_cols]
    
    print(f"Generating PID features for {len(base_cols)} base columns...")
    
    # 時間経過による重み付け (後半のデータを重視するゲイン)
    max_day = df_eng['day_n'].max()
    gain = 1.0 + (df_eng['day_n'] / max_day * 0.5)
    
    # グループ化（サンプルまたぎの計算を防ぐため）
    grouped = df_eng.groupby('sample_id')
    
    for col in base_cols:
        # --- P (Proportional: 比例) ---
        # 重み付けされた生の値
        df_eng[f'{col}_weighted'] = df_eng[col] * gain
        
        # --- I (Integral: 積分) ---
        # 累積和 (Cumsum): 蓄積された活動量
        df_eng[f'{col}_cumsum'] = grouped[col].cumsum()
        
        # --- D (Derivative: 微分) ---
        # 変化量 (Diff): 活動の変化速度
        # 1階微分
        col_diff1 = grouped[col].diff().fillna(0)
        df_eng[f'{col}_diff1'] = col_diff1 * gain
        
        # --- Lag (過去の状態) ---
        # 1つ前の時刻の値（遅延応答の考慮）
        df_eng[f'{col}_lag1'] = grouped[col].shift(1).fillna(0)
        
    return df_eng

print(">>> Loading Data...")
train = pd.read_csv(TRAIN_PATH).fillna(0)
test = pd.read_csv(TEST_PATH).fillna(0)

# 特徴量生成実行
print(">>> Engineering PID Features...")
train = add_pid_features(train)
test = add_pid_features(test)

# 特徴量カラムの選定
target_col = 'lever'
ignore_cols = ['id', 'sample_id', 'day_n', 'time', 'lever', 'mouse_id']
# 生成された全カラムを取得
train_feats = [c for c in train.columns if c not in ignore_cols]
test_feats = set(test.columns)
# Train/Test共通のカラムのみ使用
feature_cols = [c for c in train_feats if c in test_feats]

print(f"Total Input Features: {len(feature_cols)}")

# スケーリング (PID特徴量は値の範囲が広いためRobustScaler推奨)
scaler = RobustScaler()
train[feature_cols] = scaler.fit_transform(train[feature_cols])
test[feature_cols] = scaler.transform(test[feature_cols])

# ==============================================================================
# 3. Dataset & DataLoader
# ==============================================================================
# 動的に最大長を取得 (パディング用)
max_seq_len_train = train.groupby('sample_id').size().max()
max_seq_len_test = test.groupby('sample_id').size().max()
MAX_LEN = max(max_seq_len_train, max_seq_len_test)
print(f"Max Sequence Length: {MAX_LEN}")

class BrainDataset(Dataset):
    def __init__(self, df, feature_cols, target_col=None, max_len=MAX_LEN):
        self.df = df
        self.sample_ids = df['sample_id'].unique()
        self.feature_cols = feature_cols
        self.target_col = target_col
        self.max_len = max_len
        self.grouped = df.groupby('sample_id')

    def __len__(self):
        return len(self.sample_ids)

    def __getitem__(self, idx):
        sample_id = self.sample_ids[idx]
        group = self.grouped.get_group(sample_id)
        
        # 特徴量
        x = group[self.feature_cols].values
        seq_len = len(x)
        
        # Padding
        x_pad = np.zeros((self.max_len, len(self.feature_cols)), dtype=np.float32)
        x_pad[:seq_len, :] = x
        
        # Mask (データがある部分が1, パディングが0)
        mask = np.zeros((self.max_len,), dtype=np.float32)
        mask[:seq_len] = 1.0
        
        ret = {
            'x': torch.tensor(x_pad),
            'mask': torch.tensor(mask),
            'id': sample_id,
            'seq_len': seq_len
        }
        
        if self.target_col:
            y = group[self.target_col].values
            y_pad = np.zeros((self.max_len,), dtype=np.float32)
            y_pad[:seq_len] = y
            ret['y'] = torch.tensor(y_pad)
            
        return ret

# ==============================================================================
# 4. Model: 1D-CNN + LSTM (Hybrid Model)
# ==============================================================================
class PID_HybridModel(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, num_layers=2):
        super().__init__()
        
        # 1D-CNN: 局所的な特徴（微分的な急激な変化など）を捉える
        self.cnn = nn.Sequential(
            nn.Conv1d(input_dim, hidden_dim, kernel_size=3, padding=1),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Conv1d(hidden_dim, hidden_dim, kernel_size=3, padding=1),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU()
        )
        
        # LSTM: 長期的な依存関係（積分の蓄積効果など）を捉える
        self.lstm = nn.LSTM(
            input_size=hidden_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True
        )
        
        # 出力層
        self.regressor = nn.Sequential(
            nn.Linear(hidden_dim * 2, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )
        
    def forward(self, x):
        # x: [Batch, Time, Channels]
        
        # CNN用に [Batch, Channels, Time] に変換
        x = x.permute(0, 2, 1)
        x = self.cnn(x)
        
        # LSTM用に [Batch, Time, Channels] に戻す
        x = x.permute(0, 2, 1)
        x, _ = self.lstm(x)
        
        # 出力
        output = self.regressor(x)
        return output.squeeze(-1)

# ==============================================================================
# 5. Training Loop
# ==============================================================================
# 提出用IDマッピング
test_sample_map = test.groupby('sample_id')['id'].apply(list).to_dict()
submission_dict = {}

unique_samples = train['sample_id'].unique()
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
criterion = nn.MSELoss(reduction='none')

print(f"\n>>> Starting {N_FOLDS}-Fold Cross Validation...")

# Test Dataset
test_ds = BrainDataset(test, feature_cols, None, MAX_LEN)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

for fold, (train_idx, val_idx) in enumerate(kf.split(unique_samples)):
    print(f"\n=== Fold {fold+1}/{N_FOLDS} ===")
    
    train_ids, val_ids = unique_samples[train_idx], unique_samples[val_idx]
    train_ds = BrainDataset(train[train['sample_id'].isin(train_ids)], feature_cols, target_col, MAX_LEN)
    val_ds = BrainDataset(train[train['sample_id'].isin(val_ids)], feature_cols, target_col, MAX_LEN)
    
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
    
    # モデル初期化
    model = PID_HybridModel(input_dim=len(feature_cols), hidden_dim=64).to(DEVICE)
    if torch.cuda.device_count() > 1:
        model = nn.DataParallel(model)
        
    optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
    
    best_loss = float('inf')
    best_model_path = f"best_pid_dl_fold{fold}.pth"
    
    # --- Train ---
    for epoch in range(EPOCHS):
        model.train()
        train_loss = 0
        for batch in train_loader:
            x, y, m = batch['x'].to(DEVICE), batch['y'].to(DEVICE), batch['mask'].to(DEVICE)
            optimizer.zero_grad()
            preds = model(x)
            loss = (criterion(preds, y) * m).sum() / m.sum()
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
        
        scheduler.step()
        
        # --- Valid ---
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for batch in val_loader:
                x, y, m = batch['x'].to(DEVICE), batch['y'].to(DEVICE), batch['mask'].to(DEVICE)
                preds = model(x)
                loss = (criterion(preds, y) * m).sum() / m.sum()
                val_loss += loss.item()
        
        avg_val = val_loss / len(val_loader)
        
        if (epoch+1) % 5 == 0 or epoch == 0:
            print(f"  Ep {epoch+1:2d} | Train: {train_loss/len(train_loader):.4f} | Val: {avg_val:.4f}")
        
        if avg_val < best_loss:
            best_loss = avg_val
            torch.save(model.state_dict(), best_model_path)
            
    print(f"  >> Best Val: {best_loss:.5f}")
    
    # --- Inference ---
    print("  Predicting Test Data...")
    model.load_state_dict(torch.load(best_model_path))
    model.eval()
    
    with torch.no_grad():
        for batch in test_loader:
            x = batch['x'].to(DEVICE)
            s_ids = batch['id']
            seq_lens = batch['seq_len'].numpy()
            
            preds = model(x).cpu().numpy()
            
            for i, s_id in enumerate(s_ids):
                valid_len = seq_lens[i]
                pred_seq = np.maximum(0, preds[i, :valid_len])
                row_ids = test_sample_map[s_id]
                
                # 長さ補正
                if len(row_ids) != len(pred_seq):
                    min_len = min(len(row_ids), len(pred_seq))
                    row_ids = row_ids[:min_len]
                    pred_seq = pred_seq[:min_len]
                
                # 加算
                for r_id, p_val in zip(row_ids, pred_seq):
                    submission_dict[r_id] = submission_dict.get(r_id, 0) + p_val

    del model, optimizer, scheduler
    gc.collect()
    torch.cuda.empty_cache()

# ==============================================================================
# 6. Final Submission
# ==============================================================================
print("\n>>> Creating Final Submission...")

if len(submission_dict) > 0:
    results = []
    # 元のtest.csvのID順序を守る
    for r_id in test['id']:
        val = submission_dict.get(r_id, 0) / N_FOLDS
        results.append({'id': r_id, 'lever': val})

    df_sub = pd.DataFrame(results)
    df_sub.to_csv(SUBMISSION_PATH, index=False)
    print(f"SUCCESS: {SUBMISSION_PATH} created.")
else:
    print("Error: No predictions generated.")

Using Device: cpu
>>> Loading Data...
>>> Engineering PID Features...
Generating PID features for 44 base columns...
Generating PID features for 44 base columns...
Total Input Features: 220
Max Sequence Length: 40

>>> Starting 5-Fold Cross Validation...

=== Fold 1/5 ===
  Ep  1 | Train: 1.2850 | Val: 1.2990
  Ep  5 | Train: 0.8822 | Val: 1.0066
  Ep 10 | Train: 0.7888 | Val: 0.9286
  Ep 15 | Train: 0.6807 | Val: 0.9148
  Ep 20 | Train: 0.6325 | Val: 0.9053
  >> Best Val: 0.89620
  Predicting Test Data...

=== Fold 2/5 ===
  Ep  1 | Train: 1.3203 | Val: 1.2252
  Ep  5 | Train: 0.9193 | Val: 0.8721
  Ep 10 | Train: 0.8126 | Val: 0.8134
  Ep 15 | Train: 0.7122 | Val: 0.8036
  Ep 20 | Train: 0.6551 | Val: 0.8033
  >> Best Val: 0.80246
  Predicting Test Data...

=== Fold 3/5 ===
  Ep  1 | Train: 1.2918 | Val: 1.0662
  Ep  5 | Train: 0.9199 | Val: 0.9661
  Ep 10 | Train: 0.7882 | Val: 0.8532
  Ep 15 | Train: 0.6984 | Val: 0.8413
  Ep 20 | Train: 0.6431 | Val: 0.8545
  >> Best Val: 0.84132
